### Run this notebook online

[![Open in Colab](https://img.shields.io/badge/Open_in-Colab-F9AB00?logo=googlecolab&logoColor=F9AB00)](https://colab.research.google.com/github/hosein-fanai/continual_learning_with_diffusion_vision_transformers/blob/main/files/notebooks/hpo/DiT_Classifier_HPO_Runner.ipynb)
[![Open in Kaggle](https://kaggle.com/static/images/open-in-kaggle.svg)](https://www.kaggle.com/kernels/welcome?src=https%3A%2F%2Fgithub.com%2Fhosein-fanai%2Fcontinual_learning_with_diffusion_vision_transformers%2Fblob%2Fmain%2Ffiles%2Fnotebooks%2Fhpo%2FDiT_Classifier_HPO_Runner.ipynb)

- **Colab:** select a GPU runtime, then run setup. Change GPU_IDS, concurrency and WORKER_GPU_MEMORY_LIMIT_MB for the devices available.
- **Kaggle:** enable Internet and GPU acceleration before setup; adjust device selection, concurrency and the worker memory cap.
- **Remote GPU container:** open inside the matching checkout with TensorFlow **2.20.0 / Keras 3.11.2** installed. Setup verifies packages without replacing them.

Buttons open GitHub's published main revision. Publish this notebook and its matching
common APIs together before using those links. For an unpublished version, upload the
complete matching checkout and notebook to the online runtime. Hosted setup uses the
shared installer; restart if requested. Run the stages individually to review progress.

Save the complete RESULTS_PATH, including SQLite, sampler state and notebook_runner.
Resume requires the same absolute paths and source/runtime identity. Provider session
limits still apply. **No classifier capacity test has been run for this new space.**


# DiT classifier HPO: accuracy and denoising Pareto search

Train a teacher-free **DiT classifier with joint denoising**, using the native V1
diffusion_classifier wrapper. HPO receives **two separate final raw objectives**:

- Maximize **classification_accuracy**.
- Minimize **noise_loss** (MSE).

Both values guide Optuna's multi-objective sampler and define the **Pareto front**.
A configuration is dominated when another is at least as good on both objectives
and strictly better on one. There is no weighted sum or accuracy-only HPO ranking.
This is version 2 of **dit_classifier_runner**, with the archive-informed space
described in [the search design](README.md#dit-classifier-campaign-runner).

**Official test feedback, as requested:** train on all 50,000 CIFAR-10 training images.
The 10,000 official test images supply epoch validation, both final HPO objectives
and confirmation scores. These are tuning scores, not an untouched generalization estimate.

Targets are **100** finite COMPLETE trials for review, **400** for the main search,
and **450** with the optional 50-trial extension. At most **2,500 attempts** are allocated.
The persistent **20-hour** budget permits at most 18 hours of search and reserves
2 hours for confirmations. It starts on first search execution, includes admission waits
and survives restarts. Targets and all confirmations are **not guaranteed** to finish.
Every completed trial uses the same 100-epoch budget and final raw weights; single-metric
early stopping is disabled so accuracy alone does not select the evaluated checkpoint.

Default-space studies queue the archived 128-wide, 8+8-layer configuration as a fresh
parameter suggestion. No historical score or checkpoint is imported. Nonempty
SEARCH_SPACE_OVERRIDES omit this hint. Use fresh v2 results: scalar v1 studies and
their confirmation receipts cannot be resumed as two-objective studies.


## GPU selection and capacity

Run in a fresh remote Linux GPU kernel with TensorFlow **2.20.0** and Keras **3.11.2**.
The coordinator stays CPU-only; isolated admitted child processes perform training.

| GPU_IDS | CONCURRENT_TRIALS | Routing |
| --- | --- | --- |
| [0] | 1 | One trial on GPU 0 |
| [0] | 2 | Two trials on GPU 0, subject to measured memory |
| [0,1,2] | 3 | One trial on each GPU, the initial notebook setting |
| [0,1,2] | 6 | Two per GPU, only after a capacity check and a smaller worker cap |

CONCURRENT_TRIALS is the **total**, not a per-GPU value. Each trial uses one GPU;
memory is not pooled. GPU IDs bind to measured device UUIDs. A single coordinator
owns Optuna and dispatches a fresh process whenever a device slot is free.

On smaller Colab/Kaggle GPUs, reduce the worker cap or use None for the shared automatic
allocation; this also needs a workload check. The fixed 72 GiB setting cannot fit those GPUs.

The initial **73728 MiB (72 GiB)** TensorFlow cap reserves most of one 80GB device for
one classifier trial. Admission adds overhead and device headroom, preserves live jobs
and blocks unknown ownership. This is an unmeasured starting allocation, **not a
verified memory requirement or maximum concurrency**. Even one largest trial needs a
real workload check. Increasing concurrency requires measuring this classifier space and
setting a cap that fits every selected GPU; admission never silently shrinks the cap.
The generation runner's 51-worker result does not transfer to this space.

Confirmations run at most one seed per selected GPU with the same worker cap. Recognized
search OOMs are pruned and reservations released; confirmation failures remain incomplete.
Do not run this notebook on the occupied generation container. Use an available container
and recheck runtime identity, available memory and live jobs before launching.


In [ ]:
import os
import platform
import sys

from pathlib import Path
from urllib.request import urlopen


CHECKOUT_NAME = "Continual-Learning-with-Vision-Transformers"
REPOSITORY_NAME = "continual_learning_with_diffusion_vision_transformers"
REPOSITORY = f"https://github.com/hosein-fanai/{REPOSITORY_NAME}.git"
REVISION = "main"

# Reject laptop kernels before cloning a checkout or installing any packages.
if platform.system() != "Linux":
    raise RuntimeError("Use an online GPU runtime; laptop execution is prohibited.")
if os.environ.get("KAGGLE_KERNEL_RUN_TYPE"):
    RUNTIME = "kaggle"
elif os.environ.get("COLAB_RELEASE_TAG") or "google.colab" in sys.modules:
    RUNTIME = "colab"
elif Path("/.dockerenv").is_file() or Path("/run/.containerenv").is_file():
    RUNTIME = "container"
else:
    raise RuntimeError("Use Colab, Kaggle, or a remote Linux GPU container.")

# GPU work happens only in admitted children, never in this coordinator kernel.
if "tensorflow" in sys.modules or "keras" in sys.modules:
    raise RuntimeError("Restart the kernel and run setup before importing TensorFlow or Keras.")
os.environ["CUDA_VISIBLE_DEVICES"] = "-1"

os.environ["CONTINUAL_RUNTIME"] = RUNTIME
# Reuse the same standard-library initializer as the other online notebooks.
_locations = (
    Path.cwd(), *Path.cwd().parents, Path.cwd() / CHECKOUT_NAME, 
    Path("/kaggle/working") / CHECKOUT_NAME, Path("/content") / CHECKOUT_NAME
)
_initializer = next((
    path / "files" / "notebooks" / "init.py" for path in _locations
    if (path / "files" / "notebooks" / "init.py").is_file()
), None)
_url = (
    f"https://raw.githubusercontent.com/hosein-fanai/{REPOSITORY_NAME}/"
    f"{REVISION}/files/notebooks/init.py"
)
_setup = {"__name__": "notebook_setup", "__file__": str(_initializer or _url)}
with (_initializer.open("rb") if _initializer else urlopen(_url, timeout=30)) as _file:
    exec(compile(_file.read(), _setup["__file__"], "exec"), _setup)
CHECKOUT_ROOT, RUNTIME_PACKAGES = _setup["prepare_notebook"](
    checkout_name=CHECKOUT_NAME, 
    repository=REPOSITORY, 
    revision=REVISION, 
    runtime="local" if RUNTIME == "container" else RUNTIME, 
    cuda=False, 
    install=False if RUNTIME == "container" else None
)

os.chdir(CHECKOUT_ROOT)
if str(CHECKOUT_ROOT) not in sys.path:
    sys.path.insert(0, str(CHECKOUT_ROOT))
os.environ.setdefault("KERAS_HOME", str(CHECKOUT_ROOT.parent / ".keras"))

from common.dit_hpo_remote import inspect_remote
from common.dit_hpo_runner import (
    make_plan, search_summary, run_search, freeze_finalists, 
    run_confirmations, confirmation_summary, budget_summary
)


runtime = inspect_remote(CHECKOUT_ROOT)
display({key: value for key, value in runtime.items() if key not in {"source_sha256", "worker_policy", "concurrent_trials"}})
print("Fingerprinted source files:", len(runtime["source_sha256"]))

In [ ]:
DATASET = "CIFAR10"
RESULTS_PATH = "files/results/dit_classifier_hpo_v2_pareto"
VALIDATION_SOURCE = "test"
VALIDATION_RATIO = 0.0
GPU_IDS = [0, 1, 2]
CONCURRENT_TRIALS = 3  # Initial unmeasured setting: one classifier trial per GPU.
WORKER_GPU_MEMORY_LIMIT_MB = 73728  # 72 GiB; admission adds overhead/headroom.
EPOCHS = 100
N_STARTUP_TRIALS = 40
SEARCH_SEED = 42
SEARCH_PROFILE = "dit_classifier_runner"

# Empty means the complete maintained profile and its fresh archive-baseline hint.
# Narrowing a distribution requires a fresh RESULTS_PATH and omits that hint.
SEARCH_SPACE_OVERRIDES = {}

# Multi-objective HPO keeps OOM/divergence pruning but no scalar percentile pruning.
PRUNING = None

REVIEW_TARGET = 100
SEARCH_TARGET = 400
EXTENSION_TARGET = SEARCH_TARGET + 50
MAX_ATTEMPTS = 2500
BATCH_TRIALS = max(100, 4 * CONCURRENT_TRIALS)
EXPERIMENT_HOURS = 20.0
CONFIRMATION_RESERVE_HOURS = 2.0
TOP_K = 3
CONFIRMATION_SEEDS = [101, 202, 303]

plan = make_plan(
    CHECKOUT_ROOT, 
    RESULTS_PATH, 
    dataset_name=DATASET, 
    epochs=EPOCHS, 
    n_startup_trials=N_STARTUP_TRIALS, 
    concurrent_trials=CONCURRENT_TRIALS, 
    gpu_ids=GPU_IDS, 
    pruning=PRUNING, 
    validation_source=VALIDATION_SOURCE, 
    validation_ratio=VALIDATION_RATIO, 
    experiment_hours=EXPERIMENT_HOURS, 
    confirmation_reserve_hours=CONFIRMATION_RESERVE_HOURS, 
    search_space_overrides=SEARCH_SPACE_OVERRIDES, 
    worker_gpu_memory_limit_mb=WORKER_GPU_MEMORY_LIMIT_MB, 
    search_profile=SEARCH_PROFILE, 
    seed=SEARCH_SEED
)
display(plan["hpo"])
print("Persistent study:", plan["study_root"])
TENSORBOARD_DIRECTORY = Path(plan["study_root"])
print("Per-worker TensorFlow cap (MiB):", WORKER_GPU_MEMORY_LIMIT_MB)
display([{key: gpu[key] for key in ["gpu_id", "gpu_uuid", "gpu_name", "concurrent_trials", "worker_policy"]}
        for gpu in plan["identity"]["gpus"]])
print("TensorBoard root:", TENSORBOARD_DIRECTORY)
display(budget_summary(plan))


## Pruning and failed trials

**PRUNING=None** disables the scalar percentile performance pruner. The existing
Optuna report/should_prune mechanism supports a single objective; using it on accuracy
alone would not implement this two-objective request. No epoch performance scalarization
or hidden accuracy-only pruning is used.

**OOM and numerical-divergence handling remain enabled.** A recognized TensorFlow
ResourceExhaustedError or Python MemoryError prunes the affected search trial, preserves
diagnostics and releases its worker. Native NaN/Inf loss guards also prune divergent
search trials. Final evaluation requires **both** accuracy and noise loss to be finite;
a nonfinite value in either objective is pruned rather than admitted to the Pareto front.

Trials that finish use the same fixed epoch budget and final raw weights. Accuracy-only
EarlyStopping and best-accuracy restoration are disabled. Epoch validation and TensorBoard
logging remain active. Deadline cancellation is separate from scientific pruning.

Pruned/failed allocations count toward MAX_ATTEMPTS but not successful targets. Unknown
worker crashes remain errors. Confirmations use the same fixed training budget and require
both finite objectives; a failed confirmation remains incomplete and needs attention.


## Live TensorBoard

The shared training and HPO APIs write epoch metrics, hyperparameters, objective outcomes,
durations and failure diagnostics, as in 13_CIFAR10_Joint_HPO.ipynb. There is no notebook-local
training loop or duplicate event writer.

Open the dashboard before starting a search stage. **TENSORBOARD_DIRECTORY** covers
study_root/tensorboard plus confirmation event trees under notebook_runner/confirmations.
The dashboard is initially empty. This notebook uses **port 6008**; expose that port through
your runtime if using an external browser. Saved confirmation receipts and CSVs contain
the final paired-seed comparisons.


In [ ]:
from IPython import get_ipython


TENSORBOARD_DIRECTORY.mkdir(parents=True, exist_ok=True)
get_ipython().run_line_magic("load_ext", "tensorboard")
get_ipython().run_line_magic(
    "tensorboard", 
    f'--logdir "{TENSORBOARD_DIRECTORY}" --port 6008 --bind_all'
)


## Complete classifier search space

The maintained source is common/dit_classifier_hpo.py. All suggestions pass through
common.hpo.run_hpo; SEARCH_SPACE_OVERRIDES can restrict these named distributions.

| Parameter | Choices |
| --- | --- |
| dim | 64, 128, 256 |
| depth (generator) | 4, 6, 8 |
| clf_depth | 2, 4, 6, 8 |
| mha_num_heads (generator) | 4, 6, 8; native default key width |
| patch_size | 2, 4; CNN patchification |
| batch_size | 32, 64, 128 |
| optimizer | Adam, AdamW |
| learning_rate | log-uniform 0.0001–0.001 |
| weight_decay | AdamW only: log-uniform 0.000001–0.001 |
| clf_loss_coef | 0.001075, 0.00215, 0.0043, 0.0086 |
| scheduler_name | clipped_cosine, linear (diffusion schedule) |
| p_uncond | 0.1, 0.2 |
| clf_cond_type | time_label, time, None; None disables adaptive normalization |
| classifier_noise | clean, noisy32, noisy128; inference is always clean |
| classifier_readout | cls, gap_with_cls, gap_without_cls |
| classifier_dropout_rate | 0, 0.1, 0.25, 0.5 |
| clf_droppath_rate | 0, 0.05, 0.1 |
| clf_vit_block_dropout_rate | 0, 0.1 |
| clf_vit_block_attention_dropout_rate | 0, 0.1 |
| clf_vit_block_mlp_ratio | 2, 4 |
| classifier_mlp_ratio | None, 1, 2 |
| feature_aggregation | last, middle_last, all |
| classifier_route | plain, local_first, local_alternating, generator_cross_late, classifier_cross_late |

Routes use native APIs. Local variants place local mixers at classifier stage 1 or
alternating odd stages. Cross variants connect a middle generator stage or a middle
classifier stage to the final classifier block through value cross-attention. Feature
aggregation always projects to clf_dim=dim, so concatenating more sources does not
silently widen the classifier. Classifier attention has four heads; generator heads vary.

Fixed controls: raw 32px images standardized by the wrapper; teacher-free V1 joint fitting;
float32; full 1000-step generator corruption; CFG; MSE denoising with coefficient 1;
clean raw classification and raw noise MSE feedback; null-label classifier inputs; all-row supervision;
no EMA, KD, reconstruction/KL/contrastive objectives, U-shaped stages or V2 phases.
The learning-rate schedule is cosine over the fixed epoch budget. Generator FFN ratio4,
zero generator drop-path/block/attention dropout, and 2d_sincos positions remain fixed.

Clean classifier training uses the archive's modified timestep-zero recipe; noisy32/128
sample capped corruption independently of the generator's full horizon. Hidden-head ratios
are native API choices. Completed trials use the same fixed epochs and final raw weights; accuracy-only early stopping is disabled.
Archived scores motivate these choices but are not imported as new observations.


## Persistence and recovery

run_search delegates to common.hpo.run_hpo with the named profile, device assignments,
the two raw objectives, memory cap and resource context. The shared scheduler launches common.hpo_worker
processes and saves YAML configurations, histories, TensorBoard events and trial outcomes.

BATCH_TRIALS bounds each new allocation batch. Finite COMPLETE counts drive the stage
targets. Rerunning a reached stage adds no trials. SQLite, sampler state and source/runtime
identities are retained. Unknown owners block admission; existing jobs are preserved.

Use a fresh RESULTS_PATH when changing scientific settings: profile/source version,
dataset or feedback source, epochs, seed, startup count, objective pair or search distributions.
Do not repurpose generation or older joint-profile storage. GPU selection, concurrency and
memory cap are execution settings, but still require real capacity checks. Parallel
completion order can change adaptive suggestions.

The execution clock lives separately in notebook_runner/budget.json. Changing a numeric
trial target does not reset it. At cutoff, completed results remain; active attempts receive
deadline-cancellation records and are excluded from automatic interrupted-work recovery.
A brief cleanup allowance precedes phase boundaries. Finalists use only finite COMPLETE
trials. Re-running confirmations skips authenticated completed seeds and preserves old attempts.

Version 2 uses fresh files/results/dit_classifier_hpo_v2_pareto. Scalar version 1 recipes and receipts must remain separate.


In [ ]:
display(search_summary(plan))

In [ ]:
first_stage = run_search(
    plan, 
    target_completed=REVIEW_TARGET, 
    max_attempts=MAX_ATTEMPTS, 
    batch_trials=BATCH_TRIALS
)
display(first_stage)


## Review after 100 successful trials or the search cutoff

Inspect the **accuracy/noise-loss Pareto front**, classifier-route coverage, failures and
durations. Maximum accuracy and minimum noise loss may belong to different trials.
The front represents trade-offs; it does not designate one universally best configuration.
If time or attempts run out, proceed with available nondominated completed trials.


In [ ]:
import math
import optuna
import pandas as pd


database = Path(plan["study_root"]) / "study.db"
# Admission can expire before any study database is created.
if database.is_file():
    study = optuna.load_study(
        study_name=plan["study_name"], 
        storage="sqlite:///" + database.as_posix()
    )
    pareto_rows = [
        {
            "trial_number": trial.number, 
            "classification_accuracy": trial.values[0], 
            "noise_loss": trial.values[1], 
            "params": trial.params
        }
        for trial in study.best_trials
        if trial.values is not None and len(trial.values) == 2
        and all(math.isfinite(value) for value in trial.values)
    ]
    pareto_table = pd.DataFrame(pareto_rows)
    # Display order does not impose an accuracy-only objective or winner.
    if not pareto_table.empty:
        display(pareto_table.sort_values("classification_accuracy", ascending=False))
    else:
        print("No finite completed Pareto candidates yet.")
else:
    print("No study database yet; inspect search/admission status.")
display(search_summary(plan))


In [ ]:
main_stage = run_search(
    plan, 
    target_completed=SEARCH_TARGET, 
    max_attempts=MAX_ATTEMPTS, 
    batch_trials=BATCH_TRIALS
)
display(main_stage)


## Optional 50-trial extension

EXTENSION_TARGET=SEARCH_TARGET+50 adds up to 50 successful trials if time and attempts
remain. Set it equal to SEARCH_TARGET to skip the extension. Increasing the target does
not extend the persistent clock. Review Pareto coverage and both objective ranges, then
freeze representative nondominated configurations after all search workers have exited.


In [ ]:
final_stage = run_search(
    plan, 
    target_completed=EXTENSION_TARGET, 
    max_attempts=MAX_ATTEMPTS, 
    batch_trials=BATCH_TRIALS
)
display(final_stage)


## Freeze and confirm Pareto finalists

Freezing requires at least one distinct finite **nondominated** completed configuration
and no pending RUNNING/WAITING trials. TOP_K is the **maximum** number of classifier
finalists; a smaller Pareto front uses all available distinct configurations.

For a larger front, select evenly spaced entries from the accuracy-sorted front, retaining
both extremes when TOP_K is at least 2. With TOP_K=3, this keeps the highest-accuracy end,
the lowest-noise end and a middle trade-off. This is a deterministic confirmation allocation
rule, not a weighted HPO objective. TOP_K=1 explicitly chooses the highest-accuracy Pareto
candidate; keep at least 2 to compare both ends. Manifest ranks indicate display order.

Original input configurations and hashes are frozen, after which search cannot extend.
Each finalist retrains from fresh weights with [101,202,303], preserving architecture,
optimizer, losses, epoch budget and official-test feedback through the native APIs.
Dataset construction retains SEARCH_SEED; training and final evaluation use the paired seed.

Each completed receipt authenticates **both** raw objectives, their directions, the input
and seed. One confirmation per GPU may run at once, within admission and the remaining
deadline. The two-hour reserve may be insufficient. Compare both means and standard
deviations only after all required seeds finish; partial means must not declare a winner.


In [ ]:
finalists = freeze_finalists(plan, CONFIRMATION_SEEDS, top_k=TOP_K)
display(finalists)

In [ ]:
confirmation_records = run_confirmations(plan)
print("Completed confirmation runs:", len(confirmation_records))

In [ ]:
import pandas as pd


summary = pd.DataFrame(confirmation_summary(plan))
summary.to_csv(Path(plan["control_root"]) / "confirmation_summary.csv", index=False)
# Both objective means require the same complete paired seed set.
if not summary.empty:
    complete = summary[summary["all_seeds_complete"]]
    partial = summary[~summary["all_seeds_complete"]]
    # Accuracy order is only a display order for the two-objective comparison.
    if not complete.empty:
        print("Complete paired-seed results: compare mean_accuracy AND mean_noise_loss.")
        display(complete.sort_values("mean_accuracy", ascending=False))
    # Missing seeds prevent a fair final Pareto comparison across all candidates.
    if not partial.empty:
        print("Incomplete confirmations. Defer final trade-off selection until all candidates finish.")
        display(partial)
else:
    print("No confirmation results are available yet.")
display(budget_summary(plan))


## Interpreting results

Compare **mean_accuracy / std_accuracy** and **mean_noise_loss / std_noise_loss**
across the same paired fresh seeds. Higher accuracy and lower noise loss are desirable.
Search Pareto membership can change after retraining. Three repeats are a stability
check, not proof of universal superiority; no single winner is automatically declared.

Noise MSE is now an HPO objective, but remains a proxy for generation. Image quality,
diversity and continual-replay performance require separate evaluation. Official test
images participated in selection throughout.

Artifacts include study.db, trials.csv, pareto_trials.csv, configs, runs, tensorboard,
and notebook_runner with the recipe, clock, logs, frozen Pareto candidates, paired
vector receipts and confirmation summaries. Initial notebook outputs are empty.

**Validation status:** changes received source review only. Execution tests and three-H100
capacity checks remain pending an available authorized online runtime. The occupied
generation container was not contacted, altered or interrupted.
